In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    count = 0
    for filename in filenames:
        if dirname == "/kaggle/input/yahoo-qa-preprocessed" and count == 0:
            print()
            count += 1
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/artifacts/skipgram_word2vec.model.syn1neg.npy
/kaggle/input/artifacts/skipgram_embedding_matrix.npy
/kaggle/input/artifacts/glove_embedding_matrix.npy
/kaggle/input/artifacts/skipgram_word2vec.model.wv.vectors.npy
/kaggle/input/artifacts/tokenizer.pkl
/kaggle/input/artifacts/skipgram_word2vec.model

/kaggle/input/yahoo-qa-preprocessed/uptaded_train_preprocessed.csv
/kaggle/input/yahoo-qa-preprocessed/uptaded_test_preprocessed.csv


# Imports

In [3]:
import os
import joblib


import numpy as np
import pandas as pd

import pickle
import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, GRU, SimpleRNN, Bidirectional, Dense, Dropout

2025-09-09 15:54:17.036311: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1757433257.225055      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1757433257.278959      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


# NN Models with Embeddings

## 1. Load preprocessed CSVs

In [4]:
train_df = pd.read_csv("/kaggle/input/yahoo-qa-preprocessed/uptaded_train_preprocessed.csv")
test_df  = pd.read_csv("/kaggle/input/yahoo-qa-preprocessed/uptaded_test_preprocessed.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (280003, 29)
Test shape: (59999, 29)


## 2. Load tokenizer

In [5]:
with open("/kaggle/input/artifacts/tokenizer.pkl", "rb") as f:
    tokenizer = pickle.load(f)

## 4. Encode labels

In [6]:
train_texts = train_df["clean_text_basic"].astype(str).tolist()
test_texts  = test_df["clean_text_basic"].astype(str).tolist()

# Labels
y_train = train_df["Class"].values
y_test  = test_df["Class"].values

# Encode labels to integers
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train)
y_test  = label_encoder.transform(y_test)

# Save encoder
os.makedirs("dump_artifacts", exist_ok=True)

# Save label encoder
joblib.dump(label_encoder, "dump_artifacts/label_encoder.pkl")

['dump_artifacts/label_encoder.pkl']

## 4. Convert Text to Sequences and Add Padding

In [7]:
MAX_LEN = 200

X_train = tokenizer.texts_to_sequences(train_texts)
X_test  = tokenizer.texts_to_sequences(test_texts)

X_train = pad_sequences(X_train, maxlen=MAX_LEN, padding="post", truncating="post")
X_test  = pad_sequences(X_test, maxlen=MAX_LEN, padding="post", truncating="post")

print("X_train:", X_train.shape, "X_test:", X_test.shape, "y_train:", len(y_train), "y_test:", len(y_test))



X_train: (280003, 200) X_test: (59999, 200) y_train: 280003 y_test: 59999


## 5. Train/Validation split (only on train data)

In [8]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train,
    test_size=0.1,
    random_state=42,
    stratify=y_train
)

print("Train:", X_train.shape, "Validation:", X_val.shape, "Test:", X_test.shape)

Train: (252002, 200) Validation: (28001, 200) Test: (59999, 200)


## 5. Load embeddings

In [9]:
embedding_matrix_glove    = np.load("/kaggle/input/artifacts/glove_embedding_matrix.npy")
embedding_matrix_skipgram = np.load("/kaggle/input/artifacts/skipgram_embedding_matrix.npy")

print("Glove shape:", embedding_matrix_glove.shape)
print("Skipgram shape:", embedding_matrix_skipgram.shape)

Glove shape: (100000, 200)
Skipgram shape: (100000, 200)


# Utility Functions

## 2. Utility: Plot training history

## 3. Utility: Evaluate model

# Utility for DNN, SimpleRNN, GRU

In [10]:
def evaluate_model_1(model, X_test, y_test, class_names, title):
    print(f"--- Evaluation for {title} ---")
    
    # Get model predictions (probabilities)
    y_pred = model.predict(X_test)
    
    # Convert predictions from probabilities to class indices
    y_pred_classes = np.argmax(y_pred, axis=1)
    
    # --- THIS IS THE FIX ---
    # Convert one-hot encoded y_test back to class indices
    if y_test.ndim == 2: # Check if y_test is one-hot encoded
        y_test_classes = np.argmax(y_test, axis=1)
    else:
        y_test_classes = y_test # It's already in the correct format
    # ----------------------
    
    # Now, calculate metrics using the corrected y_test_classes
    acc = accuracy_score(y_test_classes, y_pred_classes)
    f1_weighted = f1_score(y_test_classes, y_pred_classes, average="weighted")
    f1_macro = f1_score(y_test_classes, y_pred_classes, average="macro")

    print(f"Accuracy: {acc:.4f}")
    print(f"F1 Score (Weighted): {f1_weighted:.4f}")
    print(f"F1 Score (Macro): {f1_macro:.4f}\n")

    # Display classification report
    print("Classification Report:")
    print(classification_report(y_test_classes, y_pred_classes, target_names=class_names))

    # Display confusion matrix
    cm = confusion_matrix(y_test_classes, y_pred_classes)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.title(f'Confusion Matrix for {title}')
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.show()


In [11]:
# --- DEBUGGING CELL ---
print(f"Shape of y_train before encoding: {y_train.shape}")
print(f"Shape of y_val before encoding: {y_val.shape}")
print(f"Shape of y_test before encoding: {y_test.shape}")

Shape of y_train before encoding: (252002,)
Shape of y_val before encoding: (28001,)
Shape of y_test before encoding: (59999,)


In [12]:
from tensorflow.keras.utils import to_categorical
import numpy as np

# Check if y_train is a 1D array before one-hot encoding
if len(y_train.shape) == 1:
    print("Labels are 1D. Performing one-hot encoding...")
    num_classes = len(np.unique(y_train))
    y_train = to_categorical(y_train, num_classes)
    y_val = to_categorical(y_val, num_classes)
    y_test = to_categorical(y_test, num_classes)
else:
    print("Labels are already one-hot encoded. Skipping encoding.")

# --- The rest of your code ---
MAX_SEQUENCE_LENGTH = X_train.shape[1]
num_classes = y_train.shape[1]
EPOCHS = 10
BATCH_SIZE = 256


def run_experiment_1(build_fn, embedding_matrix, emb_name, model_name):
    print(f"\n==== Training {model_name} with {emb_name} Embeddings ====\n")
    model = build_fn(embedding_matrix, MAX_SEQUENCE_LENGTH, num_classes)
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        verbose=1
    )
    class_names = label_encoder.classes_
    evaluate_model_1(model, X_test, y_test, class_names, f"{model_name} ({emb_name})")
    return history, model


Labels are 1D. Performing one-hot encoding...


# GRU

In [24]:
# # -----------------------------
# # GRU
# # -----------------------------
# from tensorflow.keras.layers import GRU

# def build_gru_model(embedding_matrix, max_len, num_classes, embedding_dim=200):
#     model = Sequential([
#         Embedding(
#             input_dim=embedding_matrix.shape[0],
#             output_dim=embedding_dim,
#             weights=[embedding_matrix],
#             input_length=max_len,
#             trainable=False
#         ),
#         GRU(128, return_sequences=False, dropout=0.3, recurrent_dropout=0.3),
#         Dense(64, activation="relu"),
#         Dropout(0.3),
#         Dense(num_classes, activation="softmax")
#     ])
#     model.compile(
#         loss="categorical_crossentropy",
#         optimizer="adam",
#         metrics=["accuracy"]
#     )
#     return model

## GRU (GloVe)

In [ ]:
# # Run GRU with both embeddings
# gru_history_glove, gru_glove = run_experiment_1(
#     build_gru_model, embedding_matrix_glove, "GloVe", "GRU"
# )


==== Training GRU with GloVe Embeddings ====

Epoch 1/10
798/798 ━━━━━━━━━━━━━━━━━━━━ 303s 374ms/step - accuracy: 0.2031 - loss: 2.1169 - val_accuracy: 0.6662 - val_loss: 1.0446
Epoch 2/10
798/798 ━━━━━━━━━━━━━━━━━━━━ 298s 374ms/step - accuracy: 0.6462 - loss: 1.1231 - val_accuracy: 0.6953 - val_loss: 0.9622
Epoch 3/10
798/798 ━━━━━━━━━━━━━━━━━━━━ 297s 372ms/step - accuracy: 0.6741 - loss: 1.0388 - val_accuracy: 0.7009 - val_loss: 0.9370
Epoch 4/10
689/798 ━━━━━━━━━━━━━━━━━━━━ 39s 361ms/step - accuracy: 0.6811 - loss: 1.0073

## GRU (Skip-Gram)

In [ ]:
# gru_history_skip, gru_skip = run_experiment_1(
#     build_gru_model, embedding_matrix_skipgram, "Skip-gram", "GRU"
# )

In [24]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Infer max sequence length and num classes
MAX_SEQUENCE_LENGTH = X_train.shape[1]   # padded sequence length
NUM_CLASSES = y_train.shape[1]           # one-hot encoded classes


def build_gru_model(embedding_matrix, embedding_dim=200):
    model = Sequential([
        Embedding(
            input_dim=embedding_matrix.shape[0],
            output_dim=embedding_dim,
            weights=[embedding_matrix],
            input_length=MAX_SEQUENCE_LENGTH,
            trainable=False
        ),
        GRU(
            64,
            return_sequences=False,
            dropout=0.2,
            recurrent_dropout=0.2
        ),
        Dense(32, activation="relu"),
        Dropout(0.2),
        Dense(NUM_CLASSES, activation="softmax")
    ])
    model.compile(
        loss="categorical_crossentropy",
        optimizer="adam",
        metrics=["accuracy"]
    )
    return model



# early_stop = EarlyStopping(
#     monitor="val_loss",
#     patience=3,
#     restore_best_weights=True
# )

# reduce_lr = ReduceLROnPlateau(
#     monitor="val_loss",
#     factor=0.5,
#     patience=2,
#     min_lr=1e-5
# )


from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

def train_and_evaluate_gru(embedding_matrix, name):
    model = build_gru_model(embedding_matrix)
    
    callbacks = [
        EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
        ModelCheckpoint(f"best_gru_{name}.keras", save_best_only=True, monitor="val_loss")
    ]
    
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=2,
        batch_size=256,
        callbacks=callbacks,
        verbose=1
    )
    
    print(f"\nEvaluating GRU with {name} Embeddings:\n")
    evaluate_model_1(model, X_test, y_test, label_encoder, title=f"GRU ({name})")
    
    return model, history


gru_glove, hist_glove = train_and_evaluate_gru(embedding_matrix_glove, "GloVe")
gru_skip, hist_skip   = train_and_evaluate_gru(embedding_matrix_skipgram, "Skip-gram")



Epoch 1/2
985/985 ━━━━━━━━━━━━━━━━━━━━ 394s 395ms/step - accuracy: 0.2240 - loss: 2.0709 - val_accuracy: 0.6540 - val_loss: 1.0728
Epoch 2/2
985/985 ━━━━━━━━━━━━━━━━━━━━ 393s 399ms/step - accuracy: 0.6396 - loss: 1.1400 - val_accuracy: 0.6848 - val_loss: 0.9787

Evaluating GRU with GloVe Embeddings:

--- Evaluation for GRU (GloVe) ---
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 179s 95ms/step
Accuracy: 0.6869
F1 Score (Weighted): 0.6776
F1 Score (Macro): 0.6776

Classification Report:


TypeError: object of type 'LabelEncoder' has no len()

In [23]:
print(model)

NameError: name 'model' is not defined